## 5. 向量检索

把文档片段和问题转换成向量，再用 Qdrant 找出最相关的内容。

In [1]:
import os
import re
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

load_dotenv("../.env")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
if not EMBEDDING_MODEL:
    raise RuntimeError("请在 .env 中配置 EMBEDDING_MODEL。")
model = SentenceTransformer(EMBEDDING_MODEL)


HEADER_KEYS = {1: "title", 2: "section", 3: "subsection"}


def split_by_markdown_headers(text):
    sections = []
    metadata = {}
    lines = []
    in_fence = False

    def save_section():
        content = "\n".join(lines).strip()
        body_lines = lines[1:] if lines and re.match(r"^ {0,3}#{1,3}\s+", lines[0]) else lines
        has_body = any(line.strip() and line.strip() != "---" for line in body_lines)
        if content and has_body:
            sections.append({"text": content, "metadata": metadata.copy()})

    for line in text.splitlines():
        if re.match(r"^ {0,3}(```|~~~)", line):
            in_fence = not in_fence
            lines.append(line)
            continue

        match = None if in_fence else re.match(r"^ {0,3}(#{1,3})\s+(.+)$", line)
        if not match:
            lines.append(line)
            continue

        save_section()
        lines = [line]
        level = len(match.group(1))
        metadata[HEADER_KEYS[level]] = match.group(2).strip()
        for deeper_level in range(level + 1, 4):
            metadata.pop(HEADER_KEYS[deeper_level], None)

    save_section()
    return sections

def split_long_text(text, max_chars=800, separators=("\n\n", "\n", "。", "；", "，", " ")):
    if len(text) <= max_chars:
        return [text]
    if not separators:
        return [text[index:index + max_chars] for index in range(0, len(text), max_chars)]

    separator, *remaining = separators
    if separator not in text:
        return split_long_text(text, max_chars, tuple(remaining))

    raw_parts = text.split(separator)
    parts = [
        part + separator if index < len(raw_parts) - 1 else part
        for index, part in enumerate(raw_parts)
    ]
    chunks = []
    current = ""
    for part in parts:
        candidate = current + part
        if len(candidate) <= max_chars:
            current = candidate
            continue
        if current:
            chunks.append(current)
        if len(part) <= max_chars:
            current = part
        else:
            chunks.extend(split_long_text(part, max_chars, tuple(remaining)))
            current = ""
    if current:
        chunks.append(current)
    return [chunk.strip() for chunk in chunks if chunk.strip()]


def split_markdown(text, max_chars=800):
    chunks = []
    for section in split_by_markdown_headers(text):
        for part in split_long_text(section["text"], max_chars):
            chunks.append({"text": part, "metadata": section["metadata"].copy()})
    return chunks

def load_chunks(data_dir=Path("../data")):
    items = []
    for path in sorted(data_dir.rglob("*.md")):
        if path.name == "README.md" or "images" in path.parts or "废止" in path.name:
            continue
        for chunk in split_markdown(path.read_text(encoding="utf-8")):
            missing_headings = [
                heading for heading in chunk["metadata"].values()
                if heading not in chunk["text"]
            ]
            searchable_text = "\n".join([*missing_headings, chunk["text"]])
            items.append({
                "source": path.relative_to(data_dir).as_posix(),
                "text": searchable_text,
                **chunk["metadata"],
            })
    return items


def build_index(chunks):
    vectors = model.encode([item["text"] for item in chunks], normalize_embeddings=True)
    client = QdrantClient(":memory:")
    client.create_collection(
        collection_name="fashion_knowledge",
        vectors_config=models.VectorParams(
            size=vectors.shape[1], distance=models.Distance.COSINE
        ),
    )
    client.upload_points(
        collection_name="fashion_knowledge",
        points=[
            models.PointStruct(id=i, vector=vector.tolist(), payload=chunk)
            for i, (vector, chunk) in enumerate(zip(vectors, chunks))
        ],
    )
    return client

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

In [2]:
chunks = load_chunks()
qdrant = build_index(chunks)
print(f"知识库片段：{len(chunks)}")

知识库片段：165


### 5.1. 实现向量搜索

In [3]:
def search(question, top_k=4):
    question_vector = model.encode(question, normalize_embeddings=True).tolist()
    hits = qdrant.query_points(
        collection_name="fashion_knowledge",
        query=question_vector,
        limit=top_k,
    ).points
    return [{**hit.payload, "score": hit.score} for hit in hits]

for result in search("SKU-YG301 瑜伽裤的面料成分是什么？"):
    print(f"{result['score']:.3f}  {result['source']}")
    print(result["text"][:160].replace("\n", " ") + "...")

0.841  产品/瑜伽裤-YG301/产品规格.md
瑜伽裤 SKU-YG301 技术规格书 2. 面料规格 ### 2.2 面料结构  经编(Warp-knitted)双面布：面层平纹致密组织提供防透光基础；底层微毛圈组织经碳素磨毛处理提供Butter-soft触感。克重230 GSM，厚度0.45mm，透气率85 mm/s。...
0.829  产品/瑜伽裤-YG301/产品规格.md
瑜伽裤 SKU-YG301 技术规格书 2. 面料规格 ### 2.1 纤维成分  75% Nylon 66 (锦纶/超细聚酰胺) + 25% Lycra Spandex (莱卡四面弹氨纶)。纱线40D/48F双面精密经编，克重230 GSM (±5g)。  Nylon 66熔点265°C(高于Nylon 6的220°...
0.776  产品/瑜伽裤-YG301/产品规格.md
瑜伽裤 SKU-YG301 技术规格书 4. 后整理工艺 ### 4.2 染色  经轴染色Beam Dyeing 98°C常压，活性染料Reactive Dyes。色牢度: 耐洗4级、耐汗渍4-5级、耐光4级。主色Carbon Black (Pantone Black 6 C)。...
0.771  产品/瑜伽裤-YG301/产品规格.md
瑜伽裤 SKU-YG301 技术规格书 4. 后整理工艺 ### 4.3 功能整理  - 抗菌: 银离子抗菌剂抑菌率≥99% (AATCC 100) - 防紫外线: UPF 50+ (AATCC 183) - 亲水柔软: 有机硅柔软剂表面摩擦系数≤0.15  ---...


### 5.2. 检查货号查询排名

比较不同查询下目标片段的真实排名。货号在业务上唯一，不代表它在向量检索中一定排第一。

In [4]:
def rank_of_target(question, target_source, target_keyword):
    question_vector = model.encode(question, normalize_embeddings=True).tolist()
    hits = qdrant.query_points(
        collection_name="fashion_knowledge", query=question_vector, limit=len(chunks)
    ).points
    target_rank = next(
        rank for rank, hit in enumerate(hits, start=1)
        if hit.payload["source"] == target_source and target_keyword in hit.payload["text"]
    )
    return target_rank, hits

queries = [
    "SKU-JK902",
    "SKU-JK902 Cordura 500D",
    "冲锋衣的高强度耐磨面料",
]
target_source = "产品/冲锋衣-JK902/产品规格.md"
for query in queries:
    rank, hits = rank_of_target(query, target_source, "SKU-JK902")
    print(f"查询：{query} | 目标片段排名：{rank}/{len(hits)}")
    for hit in hits[:5]:
        print(f"  {hit.score:.3f}  {hit.payload['source']}")
    print()

查询：SKU-JK902 | 目标片段排名：1/165
  0.689  产品/冲锋衣-JK902/产品规格.md
  0.682  产品/冲锋衣-JK902/产品规格.md
  0.672  产品/冲锋衣-JK902/产品规格.md
  0.670  产品/冲锋衣-JK902/产品规格.md
  0.668  产品/冲锋衣-JK902/产品规格.md

查询：SKU-JK902 Cordura 500D | 目标片段排名：1/165
  0.608  产品/冲锋衣-JK902/产品规格.md
  0.589  产品/冲锋衣-JK902/产品规格.md
  0.578  产品/冲锋衣-JK902/产品规格.md
  0.575  产品/冲锋衣-JK902/产品规格.md
  0.574  产品/冲锋衣-JK902/产品规格.md



查询：冲锋衣的高强度耐磨面料 | 目标片段排名：1/165
  0.750  产品/冲锋衣-JK902/产品规格.md
  0.745  产品/冲锋衣-JK902/产品规格.md
  0.724  产品/冲锋衣-JK902/质检报告.md
  0.719  产品/冲锋衣-JK902/产品规格.md
  0.712  产品/冲锋衣-JK902/产品规格.md



### 5.3. 区分主题相似与答案一致

向量相似度反映的是主题接近程度。下面两个片段都在讨论尺码，但结论相反。

In [5]:
size_texts = [
    "这款女式瑜伽裤版型偏大，建议通常尺码的顾客选择小一码。",
    "这款女式瑜伽裤版型偏小，建议通常尺码的顾客选择大一码。",
]
size_query = "这款瑜伽裤尺码怎么选？"
size_vectors = model.encode(size_texts, normalize_embeddings=True)
size_query_vector = model.encode(size_query, normalize_embeddings=True)
size_scores = size_vectors @ size_query_vector
for text, score in sorted(zip(size_texts, size_scores), key=lambda item: item[1], reverse=True):
    print(f"{score:.3f}  {text}")

print("相似度高说明都与尺码主题相关，但不能据此判断应该买大一码还是小一码。")

0.837  这款女式瑜伽裤版型偏大，建议通常尺码的顾客选择小一码。


0.834  这款女式瑜伽裤版型偏小，建议通常尺码的顾客选择大一码。
相似度高说明都与尺码主题相关，但不能据此判断应该买大一码还是小一码。


### 5.4. 观察实体别名与相似实体

比较实体别名与同品牌相似实体的向量得分。

In [6]:
catalog_texts = [
    "iPhone Pro4 的电池容量是 4800mAh。",
    "iPhone pro4 的电池容量是 4800mAh。",
    "苹果 Pro4 的电池容量是 4800mAh。",
    "苹果 Plus5 的电池容量是 5000mAh。",
]
catalog_query = "苹果 Pro4 的电池容量是多少？"
catalog_vectors = model.encode(catalog_texts, normalize_embeddings=True)
catalog_query_vector = model.encode(catalog_query, normalize_embeddings=True)
catalog_scores = catalog_vectors @ catalog_query_vector
for text, score in sorted(zip(catalog_texts, catalog_scores), key=lambda item: item[1], reverse=True):
    print(f"{score:.3f}  {text}")

print("稠密检索能建立 iPhone 与苹果的语义关联，但也可能召回同品牌的 Plus5。")

0.895  苹果 Pro4 的电池容量是 4800mAh。
0.871  iPhone Pro4 的电池容量是 4800mAh。
0.871  iPhone pro4 的电池容量是 4800mAh。
0.822  苹果 Plus5 的电池容量是 5000mAh。
稠密检索能建立 iPhone 与苹果的语义关联，但也可能召回同品牌的 Plus5。
